# Email classification with Jev

Each question is a Jev **choice** on OpenRouter (`typesafe/jev-1.13`). Jev returns the winning option plus a probability for every option — not generated text.

1. About a voyage? `yes` / `no` / `uncertain`
2. Which fleet vessel? A fleet name, `uncertain`, or `not_our_vessel`
3. Topics: `contract`, `schedule`, `laytime`, `position`, `other` — each `yes` / `no` / `unknown`

Needs `LLM_API_KEY` in `.env` at the repo root. Kernel: `.venv`.

**Jev cache (debug):** set `DEMURRAGE_JEV_CACHE=1` (or in the setup cell below). First run calls OpenRouter and writes `.jev_cache/`; later runs reuse cached decisions (no API). Use `DEMURRAGE_JEV_CACHE=read` to forbid new network calls.

In [1]:
import json
import os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
os.environ.setdefault("DEMURRAGE_JEV_CACHE", "1")

from src import llm
from src.emails import (
    check_topics,
    check_vessel,
    check_voyage,
    extract,
    parse_eml_file,
    review_for,
)

# os.environ[llm.MODEL_ENV] = "typesafe/jev-1.13"
fleet = json.loads(Path("fleet.json").read_text())
vessels = [v["name"] for v in fleet["vessels"]]
print("configured:", llm.is_configured(), "| model:", llm.model_name())
print("jev cache:", llm.cache_mode(), "→", llm.cache_dir())
print("fleet:", vessels)

configured: True | model: typesafe/jev-1.13
jev cache: readwrite → .jev_cache
fleet: ['KESTREL BAY', 'KESTREL POINT', 'ALBA SPIRIT', 'NORTHERN TERN', 'SILVER HERON']


In [2]:
EMAIL = "email_09.eml"

parsed = parse_eml_file(Path("emails") / EMAIL)
print(f"From: {parsed.from_address}\nDate: {parsed.sent_at}\nSubject: {parsed.subject}\n")
print(parsed.body_text)

From: Strait Marine Services <ops@straitmarine.example>
Date: 2026-02-06T05:10:40+01:00
Subject: MT KESTREL BAY - Gibraltar bunker call - sailing report

Good morning,

Sailing report for MT KESTREL BAY, bunker call Gibraltar (all times LT):

Arrived Eastern anchorage        05/02  20:00
Bunker barge alongside           05/02  21:00
Bunkering completed              06/02  03:00
Vessel sailed for Cotonou        06/02  04:30

Received: VLSFO 620 MT, MGO 60 MT. BDN signed by Chief Engineer, copy to follow
with our invoice.

ETA Cotonou as per master.

Regards,
Strait Marine Services - Gibraltar



## Call 1: about a voyage?

In [3]:
voyage = check_voyage(parsed)
print(voyage.choice, "confidence", voyage.confidence)
voyage.probabilities

yes confidence 0.98


{'yes': 0.99, 'no': 0.01, 'uncertain': 0.0}

## Call 2: which fleet vessel and voyage?

In [4]:
vessel = check_vessel(parsed, fleet)
print(vessel.choice, "confidence", vessel.confidence)
vessel.probabilities

KESTREL BAY confidence 0.99


{'KESTREL BAY': 0.99,
 'KESTREL POINT': 0.0,
 'ALBA SPIRIT': 0.0,
 'NORTHERN TERN': 0.0,
 'SILVER HERON': 0.0,
 'unknown': 0.01}

## Topics are call 3

Voyage-number identification was removed. Classification stops at vessel + topics.

In [5]:
# No voyage-number call. Vessel is enough for this stage.

## Call 3: which topics?

In [6]:
topics = check_topics(parsed)
print(topics.model_dump(exclude={"confidence"}))
topics.confidence

{'contract': 'no', 'schedule': 'yes', 'laytime': 'no', 'position': 'yes', 'other': 'no'}


{'contract': 1.0,
 'schedule': 1.0,
 'laytime': 0.64,
 'position': 0.62,
 'other': 0.91}

## Full `extract()` and review check

Same three Jev requests as the pipeline; call 1 = `no` skips calls 2 and 3.

In [7]:
extraction = extract(parsed, fleet)
print(extraction.model_dump_json(indent=2, include={
    "voyage_about", "vessel_name", "relevance", "topics", "warnings",
}))
print("review:", review_for(extraction, fleet))

{
  "vessel_name": "KESTREL BAY",
  "voyage_about": "yes",
  "relevance": "relevant",
  "topics": {
    "contract": "no",
    "schedule": "yes",
    "laytime": "no",
    "position": "yes",
    "other": "no"
  },
  "warnings": []
}
review: None


## All emails

Run once with cache enabled to populate `.jev_cache/`, then re-run this cell offline.

In [8]:
header = f"{'email':<14}{'voyage':<11}{'vessel':<16}"
header += "".join(f"{t[:8]:<10}" for t in ("contract", "schedule", "laytime", "position", "other"))
print(header + "review")
for path in sorted(Path("emails").glob("*.eml")):
    e = extract(parse_eml_file(path), fleet)
    review = review_for(e, fleet)
    row = f"{path.name:<14}{e.voyage_about:<11}{e.vessel_name or '-':<16}"
    row += "".join(f"{e.topics[t]:<10}" for t in ("contract", "schedule", "laytime", "position", "other"))
    print(row + (review.code if review else "-"))

email         voyage     vessel          contract  schedule  laytime   position  other     review
email_01.eml  yes        KESTREL BAY     yes       no        no        no        no        -
email_02.eml  yes        KESTREL BAY     no        no        no        no        no        -
email_03.eml  yes        KESTREL BAY     no        yes       yes       yes       no        -
email_04.eml  no         -               no        no        no        no        no        -
email_05.eml  yes        KESTREL BAY     yes       no        no        no        no        -
email_06.eml  yes        KESTREL BAY     no        yes       yes       no        yes       -
email_07.eml  yes        KESTREL BAY     no        no        no        no        no        -
email_08.eml  yes        KESTREL BAY     no        yes       no        yes       no        -
email_09.eml  yes        KESTREL BAY     no        yes       no        yes       no        -
email_10.eml  yes        KESTREL BAY     no        yes       yes 